# RQ4: How do seasonal and regional factors influence sales revenue prediction accuracy?
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print('Ready.')

Ready.


In [36]:
# ── LOAD DATASET ──────────────────────────────────────────────────────────────
df = pd.read_csv('marketing_and_product_performance.csv')
df = df.dropna(subset=['Revenue_Generated'])

season_col = next((c for c in df.columns if 'season' in c.lower()), None)
region_col  = next((c for c in df.columns if 'region' in c.lower()), None)
rev_col = 'Revenue_Generated'

print(f'Season col: {season_col}, Region col: {region_col}')
df.head(3)

Season col: None, Region col: None


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative
2,CMP_6SBOWN,PROD_4V8A08,22367.45,3546,265,32585.62,1.41,CUST_BR2GST,Basic,9,FLASH_J4PEON,51,160,BNDL_0HY0EF,462.87,4,Affordable


In [37]:
# ── FIGURE 4.1 — Box Plot: Revenue by Season ──────────────────────────────────
if season_col:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Figure 4.1 — Sales Revenue Distribution by Season and Region (RQ4)', fontweight='bold')

    seasons = df[season_col].unique()
    colors_s = ['#2E4057', '#048A81', '#E07A5F', '#F2CC8F']
    data_by_season = [df[df[season_col] == s][rev_col].dropna().values for s in seasons]
    bp1 = axes[0].boxplot(data_by_season, patch_artist=True, labels=[str(s) for s in seasons])
    for patch, col in zip(bp1['boxes'], colors_s):
        patch.set_facecolor(col)
    axes[0].set_title('Revenue by Season')
    axes[0].set_ylabel('Sales Revenue')
    axes[0].set_xlabel('Season')

    if region_col:
        regions = df[region_col].unique()
        colors_r = plt.cm.Set2(np.linspace(0, 1, len(regions)))
        data_by_region = [df[df[region_col] == r][rev_col].dropna().values for r in regions]
        bp2 = axes[1].boxplot(data_by_region, patch_artist=True, labels=[str(r) for r in regions])
        for patch, col in zip(bp2['boxes'], colors_r):
            patch.set_facecolor(col)
        axes[1].set_title('Revenue by Region')
        axes[1].set_ylabel('Sales Revenue')
        axes[1].set_xlabel('Region')
        plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig('Figure_4_1_Revenue_Season_Region_Boxplot.pdf', bbox_inches='tight')
    plt.show()
    print('Figure 4.1 saved.')

In [38]:
# ── FIGURE 4.2 — Heatmap: Mean Revenue by Season x Region ────────────────────
if season_col and region_col:
    pivot = df.groupby([season_col, region_col])[rev_col].mean().unstack()
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
    ax.set_title('Figure 4.2 — Mean Sales Revenue Heatmap: Season × Region (RQ4)', fontweight='bold')
    ax.set_xlabel('Region')
    ax.set_ylabel('Season')
    plt.tight_layout()
    plt.savefig('Figure_4_2_Revenue_Heatmap_Season_Region.pdf', bbox_inches='tight')
    plt.show()
    print('Figure 4.2 saved.')

In [39]:
# ── MODEL ACCURACY PER SEASON ─────────────────────────────────────────────────
def preprocess_subset(sub):
    sub = sub.copy()
    drop_cols = [c for c in sub.columns if 'id' in c.lower() or 'name' in c.lower()]
    sub.drop(columns=drop_cols, errors='ignore', inplace=True)
    for col in sub.columns:
        if sub[col].dtype == 'object':
            sub[col] = LabelEncoder().fit_transform(sub[col].astype(str))
        else:
            sub[col].fillna(sub[col].median(), inplace=True)
    return sub

rows = []
if season_col:
    for season in df[season_col].unique():
        sub = preprocess_subset(df[df[season_col] == season])
        if len(sub) < 50: continue
        X_ = sub.drop(columns=[rev_col])
        y_ = sub[rev_col]
        Xt, Xv, yt, yv = train_test_split(X_, y_, test_size=0.2, random_state=42)
        m = GradientBoostingRegressor(n_estimators=100, random_state=42)
        m.fit(Xt, yt)
        pr = m.predict(Xv)
        rows.append({'Group': f'Season: {season}', 'R²': round(r2_score(yv, pr), 4),
                     'RMSE': round(np.sqrt(mean_squared_error(yv, pr)), 2),
                     'Mean_Revenue': round(df[df[season_col]==season][rev_col].mean(), 2),
                     'Count': len(sub)})

if region_col:
    for region in df[region_col].unique():
        sub = preprocess_subset(df[df[region_col] == region])
        if len(sub) < 50: continue
        X_ = sub.drop(columns=[rev_col])
        y_ = sub[rev_col]
        Xt, Xv, yt, yv = train_test_split(X_, y_, test_size=0.2, random_state=42)
        m = GradientBoostingRegressor(n_estimators=100, random_state=42)
        m.fit(Xt, yt)
        pr = m.predict(Xv)
        rows.append({'Group': f'Region: {region}', 'R²': round(r2_score(yv, pr), 4),
                     'RMSE': round(np.sqrt(mean_squared_error(yv, pr)), 2),
                     'Mean_Revenue': round(df[df[region_col]==region][rev_col].mean(), 2),
                     'Count': len(sub)})

table_41 = pd.DataFrame(rows)
table_41.to_csv('Table_4_1_Model_Performance_Season_Region.csv', index=False)
print('Table 4.1 saved.')
table_41

Table 4.1 saved.


""


In [40]:
# ── FIGURE 4.3 — R² per Season and Region ───────────────────────────────
if len(table_41) > 0 and 'Group' in table_41.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors_bar = ['#2E4057' if 'Season' in str(g) else '#048A81' for g in table_41['Group']]
    ax.bar(table_41['Group'], table_41['R²'], color=colors_bar)
    ax.set_title('Figure 4.3 — Model R² Score by Season and Region (RQ4)', fontweight='bold')
    ax.set_ylabel('R² Score')
    ax.set_ylim(0, 1.1)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('Figure_4_3_R2_by_Season_Region.pdf', bbox_inches='tight')
    plt.show()
    print('Figure 4.3 saved.')
else:
    print('Skipping Figure 4.3 - table_41 is empty or missing Group column.')

Skipping Figure 4.3 - table_41 is empty or missing Group column.


In [41]:
print(table_41)
print(table_41.columns.tolist())

Empty DataFrame
Columns: []
Index: []
[]


## RQ4 Summary
Figures 4.1–4.3 and Table 4.1 show how revenue distributions and model accuracy vary across seasons and regions. Higher R² in certain seasons/regions indicates the model learns those patterns better. Significant gaps suggest regional or seasonal confounders not fully captured by the current features.